In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xgboost as xgb

# load data
df_deaths = pd.read_csv('../data/Deaths_1x1.txt', skiprows=2, sep=r'\s+')
df_deaths['Age'] = df_deaths['Age'].replace('110+', 110).astype(int)

df_exposure = pd.read_csv('../data/Exposures_1x1.txt', skiprows=2, sep=r'\s+')
df_exposure['Age'] = df_exposure['Age'].replace('110+', 110).astype(int)

df = df_deaths.merge(df_exposure, on=['Year', 'Age'], suffixes=('_deaths', '_exp'))
df['mx_total'] = df['Total_deaths'] / df['Total_exp']

# filter bad rows and compute log mortality
df = df[(df['Age'] <= 100) & (df['mx_total'] > 0)].copy()
df['log_mx_total'] = np.log(df['mx_total'])

# temporal train/test split — no data leakage
train_mask = df['Year'] <= 2000
X = df[['Age', 'Year']].values
y = df['log_mx_total'].values
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[~train_mask], y[~train_mask]

print(f"Data shape: {df.shape}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Any NaN/inf in y_train: {np.any(~np.isfinite(y_train))}")
print(f"Any NaN/inf in y_test: {np.any(~np.isfinite(y_test))}")

Data shape: (10201, 10)
Train: (8080, 2), Test: (2121, 2)
Any NaN/inf in y_train: False
Any NaN/inf in y_test: False


In [18]:
# xgboost regressor with early stopping
model_xgb = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

model_xgb.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)

# evaluate
y_pred_train = model_xgb.predict(X_train)
y_pred_test = model_xgb.predict(X_test)

mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print("XGBoostv1 Performance (log scale):")
print(f"Train MAE:  {mae_train:.4f}  |  Test MAE:  {mae_test:.4f}")
print(f"Train RMSE: {rmse_train:.4f}  |  Test RMSE: {rmse_test:.4f}")
print()
print("Lee-Carter Benchmark:")
print(f"MAE: 0.2367  RMSE: 0.2896")

XGBoostv1 Performance (log scale):
Train MAE:  0.1800  |  Test MAE:  0.3811
Train RMSE: 0.2417  |  Test RMSE: 0.5036

Lee-Carter Benchmark:
MAE: 0.2367  RMSE: 0.2896


In [19]:
# engineer features that help XGBoost understand trends
df['log_age'] = np.log1p(df['Age'])  # log age captures exponential mortality growth
df['age_squared'] = df['Age'] ** 2
df['age_year_interaction'] = df['Age'] * df['Year']  # captures cohort effects

X = df[['Age', 'Year', 'log_age', 'age_squared', 'age_year_interaction']].values
y = df['log_mx_total'].values

train_mask = df['Year'] <= 2000
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[~train_mask], y[~train_mask]

# refit
model_xgb2 = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

model_xgb2.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred_test2 = model_xgb2.predict(X_test)
mae_test2 = mean_absolute_error(y_test, y_pred_test2)
rmse_test2 = np.sqrt(mean_squared_error(y_test, y_pred_test2))

print("XGBoostv2 with Feature Engineering:")
print(f"Test MAE:  {mae_test2:.4f}")
print(f"Test RMSE: {rmse_test2:.4f}")
print()
print("Lee-Carter Benchmark:")
print(f"MAE: 0.2367  RMSE: 0.2896")

XGBoostv2 with Feature Engineering:
Test MAE:  0.2665
Test RMSE: 0.3366

Lee-Carter Benchmark:
MAE: 0.2367  RMSE: 0.2896


In [20]:
# build kappa_t as a feature — this encodes the time trend explicitly
# refit kappa for all years first
from sklearn.linear_model import LinearRegression

# recreate kappa_t from notebook 2
df_deaths2 = pd.read_csv('../data/Deaths_1x1.txt', skiprows=2, sep=r'\s+')
df_deaths2['Age'] = df_deaths2['Age'].replace('110+', 110).astype(int)
df_exposure2 = pd.read_csv('../data/Exposures_1x1.txt', skiprows=2, sep=r'\s+')
df_exposure2['Age'] = df_exposure2['Age'].replace('110+', 110).astype(int)
df2 = df_deaths2.merge(df_exposure2, on=['Year', 'Age'], suffixes=('_deaths', '_exp'))
df2['mx_total'] = df2['Total_deaths'] / df2['Total_exp']
df2 = df2[(df2['Age'] <= 100) & (df2['mx_total'] > 0)].copy()

matrix = df2.pivot(index='Age', columns='Year', values='mx_total')
log_mx = np.log(matrix.values)
ages = matrix.index.values
years = matrix.columns.values

alpha_x = log_mx.mean(axis=1)
centred = log_mx - alpha_x[:, np.newaxis]
U, S, Vt = np.linalg.svd(centred)
b_raw = U[:, 0]
k_raw = Vt[0, :]
beta_x = b_raw / b_raw.sum()
kappa_t = k_raw * S[0] * b_raw.sum()

# fit linear trend to kappa and extrapolate
post1950 = years >= 1950
lr = LinearRegression()
lr.fit(years[post1950].reshape(-1, 1), kappa_t[post1950])

# create kappa lookup for all years including future
all_years = np.arange(1921, 2022)
kappa_lookup = dict(zip(years, kappa_t))

# add kappa as feature to df
df['kappa_t'] = df['Year'].map(kappa_lookup)

# rebuild features
X = df[['Age', 'Year', 'log_age', 'age_squared', 'age_year_interaction', 'kappa_t']].values
y = df['log_mx_total'].values

train_mask = df['Year'] <= 2000
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[~train_mask], y[~train_mask]

model_xgb3 = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

model_xgb3.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred_test3 = model_xgb3.predict(X_test)
mae_test3 = mean_absolute_error(y_test, y_pred_test3)
rmse_test3 = np.sqrt(mean_squared_error(y_test, y_pred_test3))

print("XGBoost Hybrid (with kappa_t feature):")
print(f"Test MAE:  {mae_test3:.4f}")
print(f"Test RMSE: {rmse_test3:.4f}")
print()
print("Lee-Carter Benchmark:")
print(f"MAE: 0.2367  RMSE: 0.2896")

XGBoost Hybrid (with kappa_t feature):
Test MAE:  0.2683
Test RMSE: 0.3388

Lee-Carter Benchmark:
MAE: 0.2367  RMSE: 0.2896
